# LAB 3c:  BigQuery ML によるディープ ニューラル ネットワーク（DNN）モデル

**学習目標**

1. BigQuery ML を使用した DNN モデルの作成と評価
1. ML.TRANSFORM を用いた特徴量エンジニアリングによる DNN モデルの作成と評価
1. BigQuery の ML.PREDICT を使用した予測の計算


## はじめに 
このノートブックでは、まず特徴量エンジニアリングを行わずに、次に前のラボの特徴量エンジニアリングを使用して、赤ちゃんが生まれる前の体重を予測する複数のディープ ニューラル ネットワーク（DNN）モデルを BigQuery ML で作成します。

BigQuery ML を使用して、BigQuery の ML.TRANSFORM を使用した特徴量エンジニアリングの有無に応じた DNN モデルを作成・評価し、BigQuery の ML.PREDICT を使用して予測を計算します。復習が必要な場合は、ノートブック [BQML Baseline Model](../solutions/3a_bqml_baseline_babyweight.ipynb) でベースライン モデルの作成方法を確認するか、ノートブック [BQML Linear Models with Feature Engineering](../solutions/3b_bqml_linear_transform_babyweight.ipynb) で線形モデルと特徴量エンジニアリングをどのように組み合わせたかを確認してください。

各学習目標は、この演習用ラボ ノートブック内の __#TODO__ に対応しています。まずはこのノートブックを自身で完成させてみてから、[解説ノートブック（solution）](../solutions/3c_bqml_dnn_babyweight.ipynb) を確認してください。

## テーブルの存在確認

次のセルを実行して、データセットとデータ テーブルが以前に作成されていることを確認します。作成されていない場合は、ラボ [1b_prepare_data_babyweight](../solutions/1b_prepare_data_babyweight.ipynb) に戻って作成してください。

In [ ]:
%load_ext google.cloud.bigquery

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリであり、テーブルが存在するかどうかを確認できます。
SELECT * FROM babyweight.babyweight_data_train
LIMIT 0

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリであり、テーブルが存在するかどうかを確認できます。
SELECT * FROM babyweight.babyweight_data_eval
LIMIT 0

## ラボ課題 1: モデル 4: DNN_REGRESSOR によるモデルの複雑さの向上

DNN_REGRESSOR は、前のラボで使用してきた LINEAR_REG と比較して、新しい回帰モデルの model_type です。

* MODEL_TYPE="DNN_REGRESSOR"

* hidden_units: レイヤあたりの隠れユニット数のリスト。すべてのレイヤは全結合（fully connected）です。配列内の要素数が隠れ層の数になります。hidden_units のデフォルト値は [Min(128, N / (𝜶(Ni+No)))]（隠れ層 1 つ）です。ここで、N はトレーニング データのサイズ、Ni と No はそれぞれ入力層と出力層のユニット数、𝜶 は値 10 の定数です。このルールの制限により、モデルが過学習（オーバーフィッティング）しないようになっています。なお、現在モデル サイズには 256MB という制限があります。

* dropout: トレーニング中に特定の座標（ニューロン）をドロップする確率。ドロップアウトは、DNN における過学習を防ぐための非常に一般的な手法です。デフォルト値はゼロであり、トレーニング中に座標をドロップしないことを意味します。

* batch_size: サブイテレーションごとにネットワークをトレーニングするために渡されるサンプル数。トレーニングの速度と収束のバランスをとるため、デフォルト値は Min(1024, num_examples) です。各サブイテレーションですべてのトレーニング データを渡すと収束の問題が生じる可能性があり、推奨されません。

### DNN_REGRESSOR モデルの作成

モデルタイプを DNN_REGRESSOR を使用するように変更し、整数のリストである HIDDEN_UNITS を追加し、整数の BATCH_SIZE を追加します。
* ヒント: model_4 を作成します。

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.model_4
OPTIONS (
    # TODO: DNN のオプションを追加する
    INPUT_LABEL_COLS=["weight_pounds"],
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    # TODO: 基本特徴量とラベルを追加する
FROM
    babyweight.babyweight_data_train

### トレーニング情報の取得と評価

まずトレーニングの統計情報を見てみましょう。

In [ ]:
%%bigquery
SELECT * FROM ML.TRAINING_INFO(MODEL babyweight.model_4)

次に、評価データセットでトレーニング済みモデルを評価しましょう。

In [ ]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.model_4,
    (
    SELECT
        weight_pounds,
        is_male,
        mother_age,
        plurality,
        gestation_weeks
    FROM
        babyweight.babyweight_data_eval
    ))

評価結果の `mean_squared_error` を使用して、モデルの RMSE を計算しましょう。

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.model_4,
    (
    SELECT
        weight_pounds,
        is_male,
        mother_age,
        plurality,
        gestation_weeks
    FROM
        babyweight.babyweight_data_eval
    ))

## ラボ課題 2: 最終モデル: TRANSFORM 句の適用

前のノートブックで行ったように、予測を実行する前に特徴量セット全体を TRANSFORM 句にカプセル化する必要があります。これにより、クエリを変更することなく、トレーニングと予測の両方に同じ変換を適用できます。

最終モデルに TRANSFORM 句を適用し、クエリを実行しましょう。

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL
    babyweight.final_model

TRANSFORM(
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    # TODO: 次の FEATURE CROSS を追加する:
    # is_male, bucketed_mother_age, plurality, bucketed_gestation_weeks
)
OPTIONS (
    # TODO: DNN のオプションを追加する
    INPUT_LABEL_COLS=["weight_pounds"],
    DATA_SPLIT_METHOD="NO_SPLIT") AS

SELECT
    *
FROM
    babyweight.babyweight_data_train

まずトレーニングの統計情報を見てみましょう。

In [ ]:
%%bigquery
SELECT * FROM ML.TRAINING_INFO(MODEL babyweight.final_model)

次に、評価データセットでトレーニング済みモデルを評価しましょう。

In [ ]:
%%bigquery
SELECT
    *
FROM
    ML.EVALUATE(MODEL babyweight.final_model,
    (
    SELECT
        *
    FROM
        babyweight.babyweight_data_eval
    ))

評価結果の `mean_squared_error` を使用して、モデルの RMSE を計算しましょう。

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL babyweight.final_model,
    (
    SELECT
        *
    FROM
        babyweight.babyweight_data_eval
    ))

## ラボ課題 3: 最終モデルによる予測

モデルを評価したので、次のステップは BigQuery の `ML.PREDICT` 関数を使用して、赤ちゃんが生まれる前の体重を予測することです。

### 元のデータセットの例を使用した最終モデルからの予測

In [ ]:
%%bigquery
SELECT
    *
FROM
    ML.PREDICT(MODEL babyweight.final_model,
    (
    SELECT
        # TODO: 元のデータセットから基本特徴量の例を追加する
    ))

### シミュレーションされたデータセットの例を使用して上記の予測クエリを変更する

上記で作成した特徴量の値を使用しますが、is_male を "Unknown" に、plurality を "Multiple(2+)" に設定します。これは、性別や正確な胎数が分からない状況をシミュレートしています。

In [ ]:
%%bigquery
SELECT
    *
FROM
    ML.PREDICT(MODEL babyweight.final_model,
    (
    SELECT
        # TODO: シミュレーションされたデータセットから基本特徴量の例を追加する
    ))

## ラボのまとめ: 
このラボでは、BigQuery ML を使用して、BigQuery の ML.TRANSFORM を使用した特徴量エンジニアリングの有無に応じた DNN モデルを作成・評価し、BigQuery の ML.PREDICT を使用して予測を計算しました。

Copyright 2020 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.